In [0]:
tracks_path = "/Volumes/workspace/default/spotify_data/spotify_tracks.csv"
artists_path = "/Volumes/workspace/default/spotify_data/spotify_artists.csv"

tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(tracks_path)
)

artists_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(artists_path)
)

print("Tracks:", tracks_df.count())
print("Artists:", artists_df.count())

In [0]:
from pyspark.sql.functions import col, regexp_replace, split, explode, trim, countDistinct

genres_df = (
    artists_df
    .filter(col("genres") != "[]")
    .withColumn(
        "genres_clean",
        regexp_replace(col("genres"), r"[\[\]']", "")
    )
    .withColumn(
        "genre",
        explode(split(col("genres_clean"), ","))
    )
    .withColumn(
        "genre",
        trim(col("genre"))
    )
    .filter(col("genre") != "")
)

display(
    genres_df.select(
        "track_id",
        "name",
        "genre"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import countDistinct

top_genres_df = (
    genres_df
    .groupBy("genre")
    .agg(countDistinct("track_id").alias("track_count"))
    .orderBy(col("track_count").desc())
    .limit(15)
)

display(top_genres_df)

In [0]:
import matplotlib.pyplot as plt

top_genres_pd = top_genres_df.toPandas()

plt.figure(figsize=(10, 6))

plt.barh(
    top_genres_pd["genre"],
    top_genres_pd["track_count"]
)

plt.gca().invert_yaxis()

plt.title("Top 15 des genres musicaux dans le dataset Spotify")
plt.xlabel("Nombre de pistes")
plt.ylabel("Genre")

plt.figtext(
    0.5,
    -0.03,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
top10_genres = [
    row["genre"]
    for row in top_genres_df.limit(10).collect()
]

genre_popularity_df = (
    genres_df
    .select("track_id", "genre")
    .dropDuplicates(["track_id", "genre"])
    .filter(col("genre").isin(top10_genres))
    .join(
        tracks_df.select(
            col("id").alias("track_id"),
            "popularity"
        ),
        on="track_id",
        how="inner"
    )
)

display(genre_popularity_df.limit(20))

In [0]:
import matplotlib.pyplot as plt

genre_popularity_pd = genre_popularity_df.toPandas()

data = [
    genre_popularity_pd[
        genre_popularity_pd["genre"] == genre
    ]["popularity"]
    for genre in top10_genres
]

plt.figure(figsize=(12, 6))

plt.boxplot(
    data,
    tick_labels=top10_genres,
    showfliers=False
)

plt.title("Distribution de la popularité par genre musical")
plt.xlabel("Genre")
plt.ylabel("Popularité Spotify (0–100)")

plt.xticks(rotation=45, ha="right")

plt.figtext(
    0.5,
    -0.08,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
tracks_path = "/Volumes/workspace/default/spotify_data/spotify_tracks.csv"

tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(tracks_path)
)

print("Tracks:", tracks_df.count())

In [0]:
features_for_corr = [
    "tempo",
    "energy",
    "danceability",
    "loudness",
    "liveness",
    "valence",
    "acousticness",
    "speechiness",
    "duration_ms",
    "popularity"
]

corr_pd = (
    tracks_df
    .select(features_for_corr)
    .toPandas()
)

corr_matrix = corr_pd.corr()

corr_matrix

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

plt.imshow(corr_matrix, aspect="auto")

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr_matrix.index)),
    corr_matrix.index
)

plt.colorbar(label="Coefficient de corrélation")

plt.title("Matrice de corrélation des caractéristiques audio")

plt.tight_layout()
plt.show()

In [0]:
scatter_pd = (
    tracks_df
    .select(
        "energy",
        "valence",
        "popularity"
    )
    .toPandas()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 7))

plt.scatter(
    scatter_pd["energy"],
    scatter_pd["valence"],
    s=scatter_pd["popularity"] * 2,
    alpha=0.4
)

plt.title("Relation entre l’énergie et la valence des pistes Spotify")
plt.xlabel("Energy")
plt.ylabel("Valence")

plt.figtext(
    0.5,
    -0.03,
    "Taille des points = Popularité | Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
artists_path = "/Volumes/workspace/default/spotify_data/spotify_artists.csv"

artists_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(artists_path)
)

from pyspark.sql.functions import col, regexp_replace, split, explode, trim

genres_df = (
    artists_df
    .filter(col("genres") != "[]")
    .withColumn(
        "genres_clean",
        regexp_replace(col("genres"), r"[\[\]']", "")
    )
    .withColumn(
        "genre",
        explode(split(col("genres_clean"), ","))
    )
    .withColumn(
        "genre",
        trim(col("genre"))
    )
    .filter(col("genre") != "")
)

print("genres_df ready")

In [0]:
from pyspark.sql.functions import avg, col

selected_genres = ["pop", "rock", "edm"]

radar_df = (
    genres_df
    .select("track_id", "genre")
    .dropDuplicates(["track_id", "genre"])
    .filter(col("genre").isin(selected_genres))
    .join(
        tracks_df.select(
            col("id").alias("track_id"),
            "energy",
            "danceability",
            "valence",
            "acousticness",
            "speechiness"
        ),
        on="track_id",
        how="inner"
    )
    .groupBy("genre")
    .agg(
        avg("energy").alias("energy"),
        avg("danceability").alias("danceability"),
        avg("valence").alias("valence"),
        avg("acousticness").alias("acousticness"),
        avg("speechiness").alias("speechiness")
    )
)

display(radar_df)

In [0]:
print("Number of genres:", radar_df.count())
display(radar_df.orderBy("genre"))

In [0]:
import numpy as np
import matplotlib.pyplot as plt

radar_pd = radar_df.toPandas()

features = [
    "energy",
    "danceability",
    "valence",
    "acousticness",
    "speechiness"
]

angles = np.linspace(
    0,
    2 * np.pi,
    len(features),
    endpoint=False
).tolist()

angles += angles[:1]

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

for _, row in radar_pd.iterrows():
    values = [row[f] for f in features]
    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=2,
        label=row["genre"]
    )

    ax.fill(
        angles,
        values,
        alpha=0.1
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features)
ax.set_ylim(0, 1)

plt.title(
    "Profil audio moyen de trois genres musicaux",
    pad=20
)

plt.legend(
    loc="upper right",
    bbox_to_anchor=(1.25, 1.1)
)

plt.figtext(
    0.5,
    0.02,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.show()

In [0]:
from pyspark.sql.functions import countDistinct, col

top_artists_df = (
    artists_df
    .groupBy("name")
    .agg(
        countDistinct("track_id").alias("track_count")
    )
    .orderBy(col("track_count").desc())
    .limit(10)
)

display(top_artists_df)

In [0]:
import matplotlib.pyplot as plt

top_artists_pd = top_artists_df.toPandas()

plt.figure(figsize=(10, 6))

plt.barh(
    top_artists_pd["name"],
    top_artists_pd["track_count"]
)

plt.gca().invert_yaxis()

plt.title("Top 10 des artistes par nombre de pistes")
plt.xlabel("Nombre de pistes")
plt.ylabel("Artiste")

plt.figtext(
    0.5,
    -0.03,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
tempo_pd = (
    tracks_df
    .select("tempo")
    .filter(col("tempo") > 0)
    .toPandas()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    tempo_pd["tempo"],
    bins=40
)

plt.title("Distribution du tempo des pistes Spotify")
plt.xlabel("Tempo (BPM)")
plt.ylabel("Nombre de pistes")

plt.figtext(
    0.5,
    -0.03,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
duration_pd = (
    tracks_df
    .select("duration_ms")
    .filter(
        (col("duration_ms") >= 30000) &
        (col("duration_ms") <= 1200000)
    )
    .toPandas()
)

duration_pd["duration_min"] = duration_pd["duration_ms"] / 60000

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    duration_pd["duration_min"],
    bins=40
)

plt.title("Distribution de la durée des pistes Spotify")
plt.xlabel("Durée (minutes)")
plt.ylabel("Nombre de pistes")

plt.figtext(
    0.5,
    -0.03,
    "Source : Dataset Spotify (Kaggle)",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()

In [0]:
figures_path = "/Volumes/workspace/default/spotify_data/figures"
dbutils.fs.mkdirs(figures_path)

print("Figures folder ready")

In [0]:
from pyspark.sql.functions import (
    col, regexp_replace, split, explode, trim,
    countDistinct, avg
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# -------------------------
# 1. Load datasets
# -------------------------

tracks_path = "/Volumes/workspace/default/spotify_data/spotify_tracks.csv"
artists_path = "/Volumes/workspace/default/spotify_data/spotify_artists.csv"

tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(tracks_path)
)

artists_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(artists_path)
)


# -------------------------
# 2. Prepare genres
# -------------------------

genres_df = (
    artists_df
    .filter(col("genres") != "[]")
    .withColumn(
        "genres_clean",
        regexp_replace(col("genres"), r"[\[\]']", "")
    )
    .withColumn(
        "genre",
        explode(split(col("genres_clean"), ","))
    )
    .withColumn("genre", trim(col("genre")))
    .filter(col("genre") != "")
)


# -------------------------
# 3. Top genres
# -------------------------

top_genres_df = (
    genres_df
    .groupBy("genre")
    .agg(countDistinct("track_id").alias("track_count"))
    .orderBy(col("track_count").desc())
    .limit(15)
)

top_genres_pd = top_genres_df.toPandas()


# -------------------------
# 4. Popularity by genre
# -------------------------

top10_genres = top_genres_pd["genre"].head(10).tolist()

genre_popularity_df = (
    genres_df
    .select("track_id", "genre")
    .dropDuplicates(["track_id", "genre"])
    .filter(col("genre").isin(top10_genres))
    .join(
        tracks_df.select(
            col("id").alias("track_id"),
            "popularity"
        ),
        "track_id",
        "inner"
    )
)

genre_popularity_pd = genre_popularity_df.toPandas()


# -------------------------
# 5. Correlation
# -------------------------

features_for_corr = [
    "tempo",
    "energy",
    "danceability",
    "loudness",
    "liveness",
    "valence",
    "acousticness",
    "speechiness",
    "duration_ms",
    "popularity"
]

corr_pd = tracks_df.select(features_for_corr).toPandas()
corr_matrix = corr_pd.corr()


# -------------------------
# 6. Energy vs Valence
# -------------------------

scatter_pd = (
    tracks_df
    .select("energy", "valence", "popularity")
    .toPandas()
)


# -------------------------
# 7. Radar
# -------------------------

selected_genres = ["pop", "rock", "edm"]

radar_df = (
    genres_df
    .select("track_id", "genre")
    .dropDuplicates(["track_id", "genre"])
    .filter(col("genre").isin(selected_genres))
    .join(
        tracks_df.select(
            col("id").alias("track_id"),
            "energy",
            "danceability",
            "valence",
            "acousticness",
            "speechiness"
        ),
        "track_id",
        "inner"
    )
    .groupBy("genre")
    .agg(
        avg("energy").alias("energy"),
        avg("danceability").alias("danceability"),
        avg("valence").alias("valence"),
        avg("acousticness").alias("acousticness"),
        avg("speechiness").alias("speechiness")
    )
)

radar_pd = radar_df.toPandas()


# -------------------------
# 8. Top artists
# -------------------------

top_artists_df = (
    artists_df
    .groupBy("name")
    .agg(countDistinct("track_id").alias("track_count"))
    .orderBy(col("track_count").desc())
    .limit(10)
)

top_artists_pd = top_artists_df.toPandas()


# -------------------------
# 9. Tempo
# -------------------------

tempo_pd = (
    tracks_df
    .select("tempo")
    .filter(col("tempo") > 0)
    .toPandas()
)


# -------------------------
# 10. Duration
# -------------------------

duration_pd = (
    tracks_df
    .select("duration_ms")
    .filter(
        (col("duration_ms") >= 30000) &
        (col("duration_ms") <= 1200000)
    )
    .toPandas()
)

duration_pd["duration_min"] = duration_pd["duration_ms"] / 60000


print("✅ Toutes les données nécessaires aux figures sont prêtes.")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Top Genres
plt.figure(figsize=(10, 6))
plt.barh(top_genres_pd["genre"], top_genres_pd["track_count"])
plt.gca().invert_yaxis()
plt.title("Top 15 des genres musicaux dans le dataset Spotify")
plt.xlabel("Nombre de pistes")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig(f"{figures_path}/01_top_genres.png", dpi=300, bbox_inches="tight")
plt.close()


# 2. Popularité par genre
data = [
    genre_popularity_pd[
        genre_popularity_pd["genre"] == genre
    ]["popularity"]
    for genre in top10_genres
]

plt.figure(figsize=(12, 6))
plt.boxplot(data, tick_labels=top10_genres, showfliers=False)
plt.title("Distribution de la popularité par genre musical")
plt.xlabel("Genre")
plt.ylabel("Popularité Spotify (0–100)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{figures_path}/02_popularity_by_genre.png", dpi=300, bbox_inches="tight")
plt.close()


# 3. Matrice de corrélation
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, aspect="auto")

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr_matrix.index)),
    corr_matrix.index
)

plt.colorbar(label="Coefficient de corrélation")
plt.title("Matrice de corrélation des caractéristiques audio")
plt.tight_layout()
plt.savefig(f"{figures_path}/03_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.close()


# 4. Energy vs Valence
plt.figure(figsize=(10, 7))
plt.scatter(
    scatter_pd["energy"],
    scatter_pd["valence"],
    s=scatter_pd["popularity"] * 2,
    alpha=0.4
)

plt.title("Relation entre l’énergie et la valence des pistes Spotify")
plt.xlabel("Energy")
plt.ylabel("Valence")
plt.tight_layout()
plt.savefig(f"{figures_path}/04_energy_valence.png", dpi=300, bbox_inches="tight")
plt.close()


# 5. Radar genres
features = [
    "energy",
    "danceability",
    "valence",
    "acousticness",
    "speechiness"
]

angles = np.linspace(
    0,
    2 * np.pi,
    len(features),
    endpoint=False
).tolist()

angles += angles[:1]

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

for _, row in radar_pd.iterrows():
    values = [row[f] for f in features]
    values += values[:1]

    ax.plot(angles, values, linewidth=2, label=row["genre"])
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features)
ax.set_ylim(0, 1)

plt.title("Profil audio moyen de trois genres musicaux", pad=20)
plt.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

plt.savefig(f"{figures_path}/05_radar_genres.png", dpi=300, bbox_inches="tight")
plt.close()


# 6. Top Artists
plt.figure(figsize=(10, 6))
plt.barh(top_artists_pd["name"], top_artists_pd["track_count"])
plt.gca().invert_yaxis()

plt.title("Top 10 des artistes par nombre de pistes")
plt.xlabel("Nombre de pistes")
plt.ylabel("Artiste")
plt.tight_layout()

plt.savefig(f"{figures_path}/06_top_artists.png", dpi=300, bbox_inches="tight")
plt.close()


# 7. Distribution du tempo
plt.figure(figsize=(10, 6))
plt.hist(tempo_pd["tempo"], bins=40)

plt.title("Distribution du tempo des pistes Spotify")
plt.xlabel("Tempo (BPM)")
plt.ylabel("Nombre de pistes")
plt.tight_layout()

plt.savefig(f"{figures_path}/07_tempo_distribution.png", dpi=300, bbox_inches="tight")
plt.close()


# 8. Distribution de la durée
plt.figure(figsize=(10, 6))
plt.hist(duration_pd["duration_min"], bins=40)

plt.title("Distribution de la durée des pistes Spotify")
plt.xlabel("Durée (minutes)")
plt.ylabel("Nombre de pistes")
plt.tight_layout()

plt.savefig(f"{figures_path}/08_duration_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

print("✅ Les 8 figures ont été enregistrées.")

In [0]:
display(dbutils.fs.ls(figures_path))